In [1]:
# Cell 1: Imports and load data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

df = pd.read_csv('../data/processed/merged_clean.csv', index_col=0, parse_dates=True)

features = ['oil_lag1', 'oil_lag2', 'oil_lag3',
            'gbp_lag1', 'gbp_lag2',
            'wages_lag1', 'wages_lag3',
            'unemployment']

X = df[features]
y = df['cpi']

print("Data loaded!")
print(f"Latest observation: {df.index[-1].date()}")
print(f"\nLatest values:")
print(df[['cpi', 'oil_price', 'gbp_usd', 'wages', 'unemployment']].tail(3))

Data loaded!
Latest observation: 2026-05-01

Latest values:
            cpi   oil_price   gbp_usd     wages  unemployment
date                                                         
2026-03-01  3.3  103.134545  1.333868  3.112165           4.9
2026-04-01  2.8  117.287500  1.345405  3.112165           4.9
2026-05-01  2.8  107.139474  1.349835  3.112165           4.9


In [2]:
# Cell 2: Train rolling window model on most recent 60 months
# We use the last 60 months of data (5 years) to train
# This captures recent inflation dynamics without being anchored to pre-2021 patterns

window_size = 60

# Training window: most recent 60 months
X_train_final = X.iloc[-window_size:]
y_train_final = y.iloc[-window_size:]

# Fit model
final_model = LinearRegression()
final_model.fit(X_train_final, y_train_final)

print(f"Model trained on: {X_train_final.index[0].date()} to {X_train_final.index[-1].date()}")
print(f"Training observations: {len(X_train_final)}")

Model trained on: 2021-06-01 to 2026-05-01
Training observations: 60


In [3]:
# Cell 3: Construct the feature vector for June 2026 nowcast
# For June 2026 we need:
# oil_lag1 = May 2026 oil price
# oil_lag2 = April 2026 oil price
# oil_lag3 = March 2026 oil price
# gbp_lag1 = May 2026 GBP/USD
# gbp_lag2 = April 2026 GBP/USD

# wages_lag1 = April 2026 wages
# wages_lag3 = February 2026 wages
# unemployment = latest available

latest = df.iloc[-1]   # May 2026
prev1 = df.iloc[-2]    # April 2026
prev2 = df.iloc[-3]    # March 2026
prev3 = df.iloc[-4]    # February 2026

june_features = {
    'oil_lag1':     latest['oil_price'],   # May oil
    'oil_lag2':     prev1['oil_price'],    # April oil
    'oil_lag3':     prev2['oil_price'],    # March oil
    'gbp_lag1':     latest['gbp_usd'],     # May GBP/USD
    'gbp_lag2':     prev1['gbp_usd'],      # April GBP/USD
    'wages_lag1':   latest['wages'],       # May wages (approx)
    'wages_lag3':   prev2['wages'],        # March wages
    'unemployment': latest['unemployment'] # Latest unemployment
}

print("\nFeature values for June 2026 nowcast:")
for k, v in june_features.items():
    print(f"  {k}: {v:.3f}")


Feature values for June 2026 nowcast:
  oil_lag1: 107.139
  oil_lag2: 117.287
  oil_lag3: 103.135
  gbp_lag1: 1.350
  gbp_lag2: 1.345
  wages_lag1: 3.112
  wages_lag3: 3.112
  unemployment: 4.900


In [4]:
# Cell 4: Generate June 2026 CPI nowcast
june_X = pd.DataFrame([june_features])

# Make prediction
june_prediction = final_model.predict(june_X)[0]

# Calculate uncertainty interval using MAE from our rolling window model
# We use 1.96 * MAE as an approximate 95% confidence interval
mae_rolling = 2.22  # from notebook 04
margin = 1.96 * mae_rolling

lower = june_prediction - margin
upper = june_prediction + margin

print("=" * 45)
print("   UK CPI NOWCAST: JUNE 2026")
print("=" * 45)
print(f"   Point estimate:    {june_prediction:.2f}%")
print(f"   95% interval:      [{lower:.2f}%, {upper:.2f}%]")
print(f"   Previous CPI:      {df['cpi'].iloc[-1]:.2f}% (May 2026)")
print(f"   BoE target:        2.00%")
print(f"   Gap from target:   {june_prediction - 2.0:+.2f}pp")
print("=" * 45)

   UK CPI NOWCAST: JUNE 2026
   Point estimate:    3.57%
   95% interval:      [-0.78%, 7.92%]
   Previous CPI:      2.80% (May 2026)
   BoE target:        2.00%
   Gap from target:   +1.57pp
